# 📝 Notebook 5: Content-Based Filtering

**Mục tiêu:** Gợi phim dựa trên nội dung (genres), không cần user data

**⚠️ Chạy notebook 00 hoặc 01 trước để có data!**

In [ ]:
import subprocess, sys, os

try:
    import surprise
    import numpy as np
    import pandas as pd
    assert int(np.__version__.split('.')[0]) < 2, 'need numpy<2'
    assert int(pd.__version__.split('.')[0]) < 3, 'need pandas<3'
    print(f'✅ OK (numpy={np.__version__}, pandas={pd.__version__}, surprise={surprise.__version__})')
except Exception as e:
    print(f'📦 Installing... ({e})')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install',
        'numpy<2', 'pandas<3', 'scikit-surprise', 'scikit-learn',
        'matplotlib', 'seaborn', 'tqdm', '-q'])
    print('✅ Install xong! Runtime đang restart...')
    os.kill(os.getpid(), 9)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings('ignore')

os.makedirs('results/charts', exist_ok=True)

ratings = pd.read_csv('data/processed/ratings_clean.csv')
movies  = pd.read_csv('data/processed/movies_clean.csv')

print(f'✅ Loaded: {len(movies):,} movies')

## 2. TF-IDF trên Genres

In [ ]:
movies['genres_clean'] = movies['genres'].str.replace('|', ' ', regex=False)
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['genres_clean'])
cosine_sim = cosine_similarity(tfidf_matrix)
movie_idx = pd.Series(movies.index, index=movies['movieId'])

print(f'Matrix: {tfidf_matrix.shape}')
print(f'Vocabulary: {tfidf.get_feature_names_out()}')

In [ ]:
# Heatmap
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cosine_sim[:50, :50], cmap='Blues', ax=ax, square=True, xticklabels=False, yticklabels=False)
ax.set_title('Cosine Similarity Matrix (50 phim đầu)')
plt.tight_layout()
plt.savefig('results/charts/05_content_similarity_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Tìm phim tương tự

In [ ]:
def find_similar_movies(movie_id, cosine_sim, movies_df, movie_idx, top_n=10):
    idx = movie_idx[movie_id]
    sim_scores = sorted(enumerate(cosine_sim[idx]), key=lambda x: x[1], reverse=True)[1:top_n+1]
    return [{'title': movies_df.iloc[i]['title'], 'genres': movies_df.iloc[i]['genres'], 'similarity': round(s, 4)} for i, s in sim_scores]

toy_story = movies[movies['title'].str.startswith('Toy Story')].iloc[0]
print(f'Phim gốc: {toy_story["title"]}')
pd.DataFrame(find_similar_movies(toy_story['movieId'], cosine_sim, movies, movie_idx, 5))

## 4. Gợi ý cho User

In [ ]:
def recommend_content(user_id, ratings_df, movies_df, cosine_sim, movie_idx, top_n=10):
    user_ratings = ratings_df[ratings_df['userId'] == user_id]
    top_rated = user_ratings.nlargest(5, 'rating')
    scores = {}
    for _, row in top_rated.iterrows():
        mid = row['movieId']
        if mid in movie_idx.index:
            idx = movie_idx[mid]
            sims = cosine_sim[idx]
            for i, s in enumerate(sims):
                if i not in user_ratings['movieId'].values:
                    scores[i] = scores.get(i, 0) + s * row['rating']
    sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return [{'movieId': movies_df.iloc[idx]['movieId'], 'title': movies_df.iloc[idx]['title'], 'score': round(sc, 2)} for idx, sc in sorted_scores[:top_n]]

recs = recommend_content(1, ratings, movies, cosine_sim, movie_idx)
print('🎬 Top 10 gợi ý cho User 1 (Content-Based):')
pd.DataFrame(recs)

## 5. Tổng kết

- Content-Based giải quyết cold-start
- Explained được: "vì bạn thích X"
- Nhược: thiếu diversity → cần kết hợp Hybrid